# PointPillars on KITTI — Colab demo

Pure-PyTorch reimplementation of *PointPillars: Fast Encoders for Object Detection from Point Clouds* (Lang et al., CVPR 2019).

**Runtime → Change runtime type → GPU** before starting. Steps: install → unit tests → download KITTI → (optional) evaluate an existing checkpoint → train → evaluate → visualize.

In [ ]:
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 1. Install

In [ ]:
REPO = "https://github.com/<your-username>/pointpillars.git"   # <- update after pushing
!git clone -q $REPO /content/pointpillars || (cd /content/pointpillars && git pull -q)
%cd /content/pointpillars
!pip -q install -e ".[dev]"

## 2. Unit tests (synthetic data, ~1 min)

In [ ]:
!python -m pytest -q

## 3. KITTI

~29 GB velodyne download, 20–40 min. Only `training/{velodyne,calib,label_2}` and `ImageSets/{train,val}.txt` are needed.
If you already have KITTI on Google Drive, mount it and point `KITTI` there instead (and skip the download).

In [ ]:
KITTI = "/content/kitti"
# from google.colab import drive; drive.mount("/content/drive"); KITTI = "/content/drive/MyDrive/kitti"
!bash scripts/download_kitti.sh $KITTI

## 4. (Optional) evaluate an existing checkpoint

Upload `pointpillars.pth` (e.g. the one trained with the original notebook) to `/content/`. Bare state dicts and full checkpoints both load.

In [ ]:
import os
if os.path.exists("/content/pointpillars.pth"):
    !python tools/evaluate.py --ckpt /content/pointpillars.pth --set root=$KITTI --out-json /content/eval_old.json

## 5. Train

~2.8 min/epoch on a T4 at batch 8 (20 epochs ≈ 56 min). Save `OUT` to Drive if you want the run to survive a disconnect,
then resume with `--resume $OUT/last.pth`.

In [ ]:
OUT = "/content/outputs/car"
!python tools/train.py --cfg configs/car.yaml --out $OUT --set root=$KITTI

### Training curves

In [ ]:
import json, matplotlib.pyplot as plt
recs = [json.loads(l) for l in open(f"{OUT}/log.jsonl")]
it = [r for r in recs if r["type"] == "iter"]
val = [r for r in recs if r["type"] == "val"]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
steps = range(len(it))
for k in ("loss", "cls", "box", "dir"):
    ax[0].plot(steps, [r[k] for r in it], label=k)
ax[0].set_yscale("log"); ax[0].set_title("training loss (logged iters)"); ax[0].legend()
if val:
    ax[1].plot([r["epoch"] for r in val], [r["bev_ap@0.7"] for r in val], "o-", label="BEV AP@0.7")
    ax[1].plot([r["epoch"] for r in val], [r["3d_ap@0.7"] for r in val], "o-", label="3D AP@0.7")
    ax[1].set_title("val AP_R40 (Car)"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{OUT}/curves.png", dpi=120); plt.show()

## 6. Evaluate + export KITTI-format predictions

In [ ]:
!python tools/evaluate.py --ckpt $OUT/best_model.pth --set root=$KITTI --export $OUT/val_preds --out-json $OUT/metrics.json

## 7. Visualize (bird's-eye view)

In [ ]:
!python tools/visualize.py --ckpt $OUT/best_model.pth --set root=$KITTI --num 4 --out $OUT/bev
from IPython.display import Image, display
import glob
for p in sorted(glob.glob(f"{OUT}/bev/*.png")):
    display(Image(p, width=700))

## 8. Save results

Copy `best_model.pth`, `metrics.json`, `curves.png` and a few BEV images back to your machine (or Drive) — they go into the README results table, `assets/`, and a GitHub Release.

In [ ]:
!cd $OUT && zip -q -r /content/pointpillars_results.zip best_model.pth metrics.json curves.png log.jsonl config.yaml bev
from google.colab import files; files.download("/content/pointpillars_results.zip")